# Data Quality Checks

Before any analysis, I want to make sure the data is clean and trustworthy.
Things I'll check:

1. Total row count
2. Group split (control vs treatment)
3. Duplicate user IDs
4. Null values
5. Users in both groups (would invalidate the test)
6. Outliers in game rounds
7. Retention values are valid (should only be True/False)

If any of these fail, I need to address it before moving forward.

In [1]:
import sqlite3
import pandas as pd

DB_PATH = "../data/cookie_cats.db"
conn = sqlite3.connect(DB_PATH)

## Check 1: Total row count

Just confirming the data fully loaded. Should be 90,189.

In [2]:
query = """
SELECT COUNT(*) AS total_rows
FROM experiment_data
"""

pd.read_sql(query, conn)

,total_rows
0,90189


## Check 2: Group split

Want to confirm the split is roughly 50/50. We saw it was 49.56% / 50.44% in pandas earlier — let me verify in SQL.

In [3]:
query = """
SELECT
    version,
    COUNT(*) AS users,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
FROM experiment_data
GROUP BY version
"""

pd.read_sql(query, conn)

,version,users,pct
0,gate_30,44700,49.56
1,gate_40,45489,50.44


## Check 3: Duplicate user IDs

Each user should appear exactly once. If any user appears twice, the data is corrupt.

In [4]:
query = """
SELECT
    userid,
    COUNT(*) AS times_appearing
FROM experiment_data
GROUP BY userid
HAVING COUNT(*) > 1
"""

result = pd.read_sql(query, conn)
print(f"Number of duplicate users: {len(result)}")
result.head()

Number of duplicate users: 0


,userid,times_appearing


## Check 4: Null values

Checking each column for any nulls. The dataset should be fully populated.

In [7]:
query = """
SELECT
    SUM(CASE WHEN userid IS NULL THEN 1 ELSE 0 END) AS null_userid,
    SUM(CASE WHEN version IS NULL THEN 1 ELSE 0 END) AS null_version,
    SUM(CASE WHEN sum_gamerounds IS NULL THEN 1 ELSE 0 END) AS null_sum_gamerounds,
    SUM(CASE WHEN retention_1 IS NULL THEN 1 ELSE 0 END) AS null_retention_1,
    SUM(CASE WHEN retention_7 IS NULL THEN 1 ELSE 0 END) AS null_retention_7
FROM experiment_data
"""
pd.read_sql(query, conn)

,null_userid,null_version,null_sum_gamerounds,null_retention_1,null_retention_7
0,0,0,0,0,0


## Check 5: Users in both groups

A user should only see one variant. If anyone appears in both gate_30 and gate_40, the test is invalid.

In [8]:
query = """
SELECT
    userid,
    COUNT(DISTINCT version) AS num_versions
FROM experiment_data
GROUP BY userid
HAVING COUNT(DISTINCT version) > 1
"""
result = pd.read_sql(query, conn)
print(f"Users in both groups: {len(result)}")
result.head()

Users in both groups: 0


,userid,num_versions


## Check 6: Game rounds distribution

The earlier pandas check showed one user played 49,854 rounds, which is suspicious. Let me look at the distribution by group.

In [9]:
query = """
SELECT
    version,
    MIN(sum_gamerounds) AS min_rounds,
    MAX(sum_gamerounds) AS max_rounds,
    ROUND(AVG(sum_gamerounds), 2) AS avg_rounds,
    COUNT(*) AS total_users
FROM experiment_data
GROUP BY version
"""
pd.read_sql(query, conn)

,version,min_rounds,max_rounds,avg_rounds,total_users
0,gate_30,0,49854,52.46,44700
1,gate_40,0,2640,51.30,45489


In [10]:
query = """
SELECT *
FROM experiment_data
WHERE sum_gamerounds > 5000
ORDER BY sum_gamerounds DESC
"""
pd.read_sql(query, conn)

,userid,version,sum_gamerounds,retention_1,retention_7
0,6390605,gate_30,49854,0,1


## Check 7: Retention column validity

retention_1 and retention_7 should only contain True/False. Checking nothing weird snuck in.

In [11]:
query = """
SELECT
    retention_1,
    retention_7,
    COUNT(*) AS users
FROM experiment_data
GROUP BY retention_1, retention_7
"""
pd.read_sql(query, conn)

,retention_1,retention_7,users
0,0,0,46437
1,0,1,3599
2,1,0,26971
3,1,1,13182


In [12]:
conn.close()
print("Done. Connection closed.")

Done. Connection closed.


## Summary of findings

Quick notes on what I saw running these checks:

**The good:**
- All 90,189 rows loaded correctly
- No nulls in any column — fully populated dataset
- No duplicate user IDs
- No users assigned to both groups (test integrity is intact)
- Retention values are clean True/False, no surprises

**Group split:**
- gate_30: 44,700 users (49.56%)
- gate_40: 45,489 users (50.44%)
- Slight imbalance but well within normal range. I'll formally test this with a chi-square (SRM check) in the analysis phase.

**The concern: outliers in game rounds**
- Most users played a reasonable number of rounds (median around 16-17)
- But one user in gate_30 played **49,854 rounds**. That's almost certainly bad data — likely a bot, test account, or logging error
- gate_40's max was 2,640, which is high but plausible for a hardcore player
- A handful of other users played > 5,000 rounds

**What I'll do about the outlier:**
I'll handle this in the EDA / analysis phase. Two options to consider:
1. Remove users above some threshold (e.g., > 99th percentile)
2. Use a robust statistic like median, or log-transform the data, instead of mean

I'll decide which after looking at the distribution more carefully.

**Bottom line:** The data is clean enough to proceed with analysis. The one outlier needs handling but doesn't invalidate anything.